In [1]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import json, glob, os
from PIL import Image
import scib

import anndata as ad

import matplotlib.pyplot as plt
import matplotlib.colors as colors
from datetime import datetime   


from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

import seaborn as sns
sns.set_theme(style='white')

#from statsmodels.stats.multitest import fdrcorrection
from tqdm import tqdm

from scipy.stats import zscore, spearmanr

import warnings
warnings.filterwarnings("ignore")



# Path to one of your spatial binning levels
base_dir = "./Prostate_spatial_data"

# finding unique samplle prefixes
samples = sorted(set(
    "_".join(os.path.basename(f).split("_")[:3])
    for f in glob.glob(os.path.join(base_dir, "*_matrix.mtx"))
))

print(samples)

adatas = {}

for sample in samples:
    print(f"Loading {sample} ...")
    prefix = os.path.join(base_dir, sample)
    
    # Read 10x count matrix
    ad = sc.read_mtx(prefix + "_matrix.mtx").T
    ad.var_names = pd.read_csv(prefix + "_features.tsv", sep="\t", header=None)[1]
    ad.obs_names = pd.read_csv(prefix + "_barcodes.tsv", sep="\t", header=None)[0]
    
    #  Load spatial coordinates 
    pos = pd.read_csv(prefix + "_tissue_positions_list.csv", header=None)
    pos.columns = ["barcode", "in_tissue", "array_row", "array_col", "pxl_row_in_fullres", "pxl_col_in_fullres"]
    pos.index = pos["barcode"]

    #  keep only barcodes that exist in ad.obs_names
    pos = pos.loc[pos["barcode"].isin(ad.obs_names), :]

    # ensure ordering matches AnnData obs
    pos = pos.reindex(ad.obs_names)

    # Join metadata and assign spatial coordinates
    ad.obs = ad.obs.join(pos, how="left")
    ad.obsm["spatial"] = pos[["pxl_row_in_fullres", "pxl_col_in_fullres"]].to_numpy(dtype=float)

    # --- Load scalefactors ---
    with open(prefix + "_scalefactors_json.json") as f:
        ad.uns["spatial"] = {sample: {"scalefactors": json.load(f)}}
    
    # Load images 
    img = Image.open(prefix + "_tissue_lowres_image.png")
    ad.uns["spatial"][sample]["images"] = {"lowres": np.array(img)}
    
    ad.uns["spatial"][sample]["metadata"] = {"source": sample}
    
    adatas[sample] = ad

print(f"\nLoaded {len(adatas)} spatial samples.")


#color dictationary for annotations
color_dict = {
 'Tumor': '#fc8d62',
 'Luminal epithelium': '#8da0cb',
 'Basal epithelium': '#66c2a5',
 'Club epithelium': '#ffd92f',
 'Immune': '#a6d854',
 'Endothelium': '#e78ac3',
 'Fibroblast': '#e5c494',
 'Muscle': '#b3b3b3'
 }

regions = list(color_dict.keys())
region_colors = list(color_dict.values())

/Users/ebouvet/Desktop/Prostate_spatial/.venv/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/ebouvet/Desktop/Prostate_spatial/.venv/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/Users/ebouvet/Desktop/Prostate_spatial/.venv/lib/python3.12/site-packages/squidpy/gr/_utils.py:23: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  CAN_US

['GSM8557976_BPH_1', 'GSM8557977_BPH_2', 'GSM8557978_BPH_3', 'GSM8557979_BPH_4', 'GSM8557980_TRNA_1', 'GSM8557981_TRNA_2', 'GSM8557982_TRNA_3', 'GSM8557983_TRNA_4', 'GSM8557984_TRNA_5', 'GSM8557985_TRNA_6', 'GSM8557986_TRNA_7', 'GSM8557987_TRNA_8', 'GSM8557988_TRNA_9', 'GSM8557989_TRNA_10', 'GSM8557990_TRNA_11', 'GSM8557991_TRNA_12', 'GSM8557992_TRNA_13', 'GSM8557993_TRNA_14', 'GSM8557994_TRNA_15', 'GSM8557995_TRNA_16', 'GSM8557996_TRNA_17', 'GSM8557997_NEADT_1', 'GSM8557998_NEADT_2', 'GSM8557999_NEADT_3', 'GSM8558000_NEADT_4', 'GSM8558001_NEADT_5', 'GSM8558002_NEADT_6', 'GSM8558003_NEADT_7', 'GSM8558004_NEADT_8', 'GSM8558005_NEADT_9', 'GSM8558006_NEADT_10', 'GSM8558007_NEADT_11', 'GSM8558008_NEADT_12', 'GSM8558009_NEADT_13', 'GSM8558010_NEADT_14', 'GSM8558011_NEADT_15', 'GSM8558012_NEADT_16', 'GSM8558013_NEADT_17', 'GSM8558014_NEADT_18', 'GSM8558015_NEADT_19', 'GSM8558016_NEADT_20', 'GSM8558017_NEADT_21', 'GSM8558018_NEADT_22', 'GSM8558019_CRPC_1', 'GSM8558020_CRPC_2', 'GSM8558021_CRP

In [5]:
def qc_and_normalize(adata):
    # QC
    sc.pp.filter_genes(adata, min_cells=5)
    sc.pp.filter_cells(adata, min_counts=500)

    # Save raw counts before normalization
    adata.layers["raw"] = adata.X.copy()

    # Normalize X
    sc.pp.normalize_total(adata, target_sum=1e4)

    # Create logp layer (log1p of normalized counts)
    adata.layers["logp"] = sc.pp.log1p(adata.X.copy())

    return adata


In [ ]:
for sample in tqdm(samples, desc="Processing sample", unit="sample"):
    slide = adatas[sample]
    print(slide.layers)

    arr = slide.X

    # Convert sparse → dense for quick inspection
    if hasattr(arr, "todense"):
        arr = np.array(arr[:5, :5].todense())
    else:
        arr = arr[:5, :5]

    print(arr)

    X = slide.X
    if hasattr(X, "A"):  # sparse
        X = X.A

    print("Global max:", X.max())
    print("Global min:", X.min())
    print("Nonzero entries:", (X > 0).sum())

    









In [6]:
for sample in samples:
    adatas[sample] = qc_and_normalize(adatas[sample])

outdir = "./processed_h5ads"
os.makedirs(outdir, exist_ok=True)

for sample, ad in adatas.items():
    ad.write_h5ad(os.path.join(outdir, f"{sample}_processed.h5ad"))

